In [ ]:

# simple_yolov8_evaluation.py
import time
import json
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import torch
from tqdm import tqdm
from datetime import datetime

def evaluate_yolov8_widerperson(model_path, data_yaml, output_dir="results"):
    """
    Simple evaluation of YOLOv8 on WiderPerson dataset.
    Returns mAP, AP, FPS, and inference time metrics.
    """
    
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    device = 0 if torch.cuda.is_available() else 'cpu'
    model = YOLO(model_path)
    
    print("="*60)
    print("YOLOv8 EVALUATION ON WIDERPERSON DATASET")
    print("="*60)
    print(f"Model: {model_path}")
    print(f"Device: {device}")
    print(f"Dataset: {data_yaml}")
    print("="*60)
    
    # 1. VALIDATION METRICS (mAP, AP, etc.)
    print("\n1. Running validation for mAP metrics...")
    val_results = model.val(
        data=data_yaml,
        imgsz=480,
        batch=16,
        device=device,
        conf=0.001,  # Low threshold for accurate mAP calculation
        iou=0.5,
        verbose=True,
        split='val',
        task='detect'
    )
    
    # Extract metrics
    metrics = {
        'mAP50': float(val_results.box.map50),
        'mAP50-95': float(val_results.box.map),
        'mAP75': float(val_results.box.map75),
        'precision': float(val_results.box.mp),
        'recall': float(val_results.box.mr),
        'val_speed': {
            'preprocess_ms': val_results.speed['preprocess'],
            'inference_ms': val_results.speed['inference'], 
            'postprocess_ms': val_results.speed['postprocess'],
            'total_ms': sum(val_results.speed.values())
        }
    }
    
       # 2. INFERENCE SPEED TEST (FPS and timing)
    print("\n2. Testing inference speed...")

    # ========================================================================= #
    # START OF THE FIX
    # ========================================================================= #
    dataset_root = Path(data_yaml).parent
    test_list_path = dataset_root / 'test.txt'

    print(f"Loading test images from: {test_list_path}")

    with open(test_list_path, 'r') as f:
        # Read relative paths from test.txt
        relative_paths = [line.strip() for line in f if line.strip()]

    # Resolve relative paths to full, absolute paths
    test_images = [str(dataset_root / p) for p in relative_paths]

    # Sanity check: ensure images actually exist
    existing_images = [p for p in test_images if Path(p).exists()]
    if len(existing_images) < len(test_images):
        print(f"Warning: {len(test_images) - len(existing_images)} images from test.txt not found and will be skipped.")
    if not existing_images:
        raise FileNotFoundError(f"No test images found. Check paths in {test_list_path} and contents of {dataset_root / 'images'}")
    
    test_images = existing_images # Use only the images that exist
    # ========================================================================= #
    # END OF THE FIX
    # ========================================================================= #


    # Warmup
    print("Warming up...")
    for _ in range(10):
        _ = model(test_images[0], imgsz=480, device=device, verbose=False)

    # Measure inference time
    print(f"Measuring inference time on {len(test_images)} test images...")
    inference_times = []

    # Use a smaller sample for speed test if the test set is huge
    sample_size = min(1000, len(test_images))
    for img_path in tqdm(test_images[:sample_size], desc="Inference"):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        start_time = time.perf_counter()
        _ = model(img_path, imgsz=640, device=device, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        inference_time = time.perf_counter() - start_time
        inference_times.append(inference_time)

    # Calculate statistics
    if inference_times:
        inference_times_ms = np.array(inference_times) * 1000  # Convert to ms
        mean_time_ms = float(np.mean(inference_times_ms))
        metrics['inference_speed'] = {
            'num_images': len(inference_times_ms),
            'mean_time_ms': mean_time_ms,
            'std_time_ms': float(np.std(inference_times_ms)),
            'min_time_ms': float(np.min(inference_times_ms)),
            'max_time_ms': float(np.max(inference_times_ms)),
            'percentiles': {
                '50th': float(np.percentile(inference_times_ms, 50)),
                '90th': float(np.percentile(inference_times_ms, 90)),
                '95th': float(np.percentile(inference_times_ms, 95)),
                '99th': float(np.percentile(inference_times_ms, 99))
            },
            'fps': float(1000 / mean_time_ms) if mean_time_ms > 0 else 0
        }
    else:
        metrics['inference_speed'] = {}
    
    # 3. GENERATE TEST PREDICTIONS
    print("\n3. Generating test set predictions...")
    pred_dir = Path(data_yaml).parent.parent / 'Evaluation' / f'yolov8_predictions_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    pred_dir.mkdir(parents=True, exist_ok=True)
    
    total_detections = 0
    for img_path in tqdm(test_images, desc="Generating predictions"):
        results = model(img_path, imgsz=640, conf=0.001, device=device, verbose=False)
        
        # Save in WiderPerson format
        img_name = Path(img_path).stem
        pred_file = pred_dir / f"{img_name}.txt"
        
        with open(pred_file, 'w') as f:
            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes
                xyxy = boxes.xyxy.cpu().numpy()
                confs = boxes.conf.cpu().numpy()
                
                f.write(f"{len(xyxy)}\n")
                total_detections += len(xyxy)
                
                for box, conf in zip(xyxy, confs):
                    x1, y1, x2, y2 = box
                    f.write(f"{x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f} {conf:.6f}\n")
            else:
                f.write("0\n")
    
    metrics['predictions'] = {
        'output_dir': str(pred_dir),
        'total_detections': total_detections,
        'avg_detections_per_image': total_detections / len(test_images)
    }
    
    # 4. PRINT SUMMARY
    print("\n" + "="*60)
    print("EVALUATION RESULTS SUMMARY")
    print("="*60)
    
    print("\nDetection Performance:")
    print(f"  mAP@50:      {metrics['mAP50']:.4f}")
    print(f"  mAP@50-95:   {metrics['mAP50-95']:.4f}")
    print(f"  mAP@75:      {metrics['mAP75']:.4f}")
    print(f"  Precision:   {metrics['precision']:.4f}")
    print(f"  Recall:      {metrics['recall']:.4f}")
    
    print("\nInference Speed:")
    print(f"  Average FPS: {metrics['inference_speed']['fps']:.2f}")
    print(f"  Average inference time: {metrics['inference_speed']['mean_time_ms']:.2f}ms ± {metrics['inference_speed']['std_time_ms']:.2f}ms")
    print(f"  Min time: {metrics['inference_speed']['min_time_ms']:.2f}ms")
    print(f"  Max time: {metrics['inference_speed']['max_time_ms']:.2f}ms")
    print(f"  95th percentile: {metrics['inference_speed']['percentiles']['95th']:.2f}ms")
    
    print("\nTest Set Predictions:")
    print(f"  Output directory: {metrics['predictions']['output_dir']}")
    print(f"  Total detections: {metrics['predictions']['total_detections']}")
    print(f"  Avg detections/image: {metrics['predictions']['avg_detections_per_image']:.1f}")
    
    # 5. SAVE RESULTS
    results_file = output_dir / f"evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(results_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    print(f"\nResults saved to: {results_file}")
    
    # 6. CREATE SIMPLE REPORT
    report = f"""# YOLOv8 WiderPerson Evaluation Results

**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Model:** {model_path}  
**Device:** {device}  

## Detection Performance (Validation Set)
- **mAP@50:** {metrics['mAP50']:.4f}
- **mAP@50-95:** {metrics['mAP50-95']:.4f}
- **mAP@75:** {metrics['mAP75']:.4f}
- **Precision:** {metrics['precision']:.4f}
- **Recall:** {metrics['recall']:.4f}

## Inference Speed (Test Set)
- **Average FPS:** {metrics['inference_speed']['fps']:.2f}
- **Average inference time:** {metrics['inference_speed']['mean_time_ms']:.2f}ms
- **Standard deviation:** {metrics['inference_speed']['std_time_ms']:.2f}ms
- **95th percentile:** {metrics['inference_speed']['percentiles']['95th']:.2f}ms

## Speed Breakdown (Validation)
- **Preprocessing:** {metrics['val_speed']['preprocess_ms']:.2f}ms
- **Inference:** {metrics['val_speed']['inference_ms']:.2f}ms  
- **Postprocessing:** {metrics['val_speed']['postprocess_ms']:.2f}ms
- **Total:** {metrics['val_speed']['total_ms']:.2f}ms

## Test Predictions
- **Output directory:** {metrics['predictions']['output_dir']}
- **Total detections:** {metrics['predictions']['total_detections']}
- **Average detections per image:** {metrics['predictions']['avg_detections_per_image']:.1f}
"""
    
    report_file = output_dir / f"evaluation_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    with open(report_file, 'w') as f:
        f.write(report)
    
    print(f"Report saved to: {report_file}")
    
    return metrics

In [ ]:
!rm /home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean/labels.cache
if __name__ == "__main__":
    # Configuration

    models = [
        "yolo12n","yolo12s","yolo12m","yolo12m","yolo12l", "yolo12x"
    ]
    for i in models:
        MODEL_PATH = f"{i}.engine"  # or path to your trained model
        DATA_YAML = "/home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean/dataset.yaml"
        OUTPUT_DIR = f"{i}_widerperson_results"
        
        # Run evaluation
        results = evaluate_yolov8_widerperson(MODEL_PATH, DATA_YAML, OUTPUT_DIR)

In [2]:
from pathlib import Path

if __name__ == "__main__":
    # Configuration
    DATA_YAML = "/home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean/dataset.yaml"
    SEARCH_DIR = Path("/home/ika1/yzlm/Re-id/object_detection_Re-ID/WiderPersonYOLO/TensorRtModels")  # change to Path("/path/to/dir") if needed

    # Find all .engine files in the directory
    engine_files = sorted(SEARCH_DIR.glob("*.engine"))

    if not engine_files:
        print(f"No .engine files found in {SEARCH_DIR.resolve()}")
    else:
        print("Found .engine files:")
        for p in engine_files:
            print(f" - {p.name}")

        # Run evaluation for each .engine file
        for engine_path in engine_files:
            MODEL_PATH = str(engine_path)
            model_name = engine_path.stem  # filename without extension
            OUTPUT_DIR = f"{model_name}_tensorrt_results"

            print(f"Evaluating {MODEL_PATH} -> {OUTPUT_DIR}")
            try:
                results = evaluate_yolov8_widerperson(MODEL_PATH, DATA_YAML, OUTPUT_DIR)
            except Exception as e:
                print(f"Error evaluating {MODEL_PATH}: {e}")
            break

Found .engine files:
 - yolo11l.engine
 - yolo11m.engine
 - yolo11n.engine
 - yolo11s.engine
 - yolo11x.engine
 - yolo12l.engine
 - yolo12m.engine
 - yolo12n.engine
 - yolo12s.engine
 - yolov10l.engine
 - yolov10m.engine
 - yolov10n.engine
 - yolov10s.engine
 - yolov10x.engine
 - yolov8l.engine
 - yolov8m.engine
 - yolov8n.engine
 - yolov8s.engine
 - yolov8x.engine
 - yolov9c.engine
 - yolov9m.engine
 - yolov9s.engine
 - yolov9t.engine
Evaluating /home/ika1/yzlm/Re-id/object_detection_Re-ID/WiderPersonYOLO/TensorRtModels/yolo11l.engine -> yolo11l_tensorrt_results
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
YOLOv8 EVALUATION ON WIDERPERSON DATASET
Model: /home/ika1/yzlm/Re-id/object_detection_Re-ID/WiderPersonYOLO/TensorRtModels/yolo11l.engine
Device: 0
Dataset: /home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean/dataset.yaml

1. Running validation fo

val: Scanning /home/ika1/yzlm/Re-id/object_detection_Re-ID/WP_YOLO_Clean/labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1000/1000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  98%|█████████▊| 62/63 [00:09<00:00,  6.70it/s]

Error evaluating /home/ika1/yzlm/Re-id/object_detection_Re-ID/WiderPersonYOLO/TensorRtModels/yolo11l.engine: input size torch.Size([8, 3, 480, 480]) not equal to max model size (16, 3, 480, 480)
